# F5-TTS Child Voice Fine-Tuning / 童声微调

Fine-tune F5-TTS v1 Base on 224 short Mandarin child-speech clips (~14 min).

Pipeline: **data check → smoke test → full training → checkpoint selection → inference & evaluation**.

> Read `docs/TRAINING_RUNBOOK.md` first. Failure history and root causes: `docs/DEBUG_NOTES.md`.

All commands assume the `f5-tts` conda env and run from the repo root.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "src" / "f5_tts").exists(), "Start Jupyter from the repo root!"
print("Repo root:", ROOT)

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Step 1: Data / 数据准备

The training text must be **pinyin token lists** (the pretrained vocab is pinyin-syllable based;
raw Chinese characters would all be out-of-vocab).
If `data/child-tts_pinyin/` does not exist, re-tokenize the existing transcripts (no Whisper re-run needed).

For a fresh dataset, run `python src/f5_tts/train/datasets/data_prepare.py <wav_dir> <out_dir>` instead
(it transcribes with Whisper and converts to pinyin in one pass).

In [ ]:
if not (ROOT / "data" / "child-tts_pinyin" / "raw.arrow").exists():
    !python -X utf8 src/f5_tts/train/datasets/retokenize_to_pinyin.py \
        --src data/child-tts_processed --dst data/child-tts_pinyin --wav-dir data/child-tts
else:
    print("data/child-tts_pinyin already exists, skip.")

# Sanity check: every pinyin token in the dataset must exist in the pretrained vocab
pretrained = set(open("ckpts/child-tts/vocab.txt", encoding="utf-8").read().splitlines())
used = set(open("data/child-tts_pinyin/vocab.txt", encoding="utf-8").read().splitlines())
print(f"tokens used: {len(used)}, missing from pretrained vocab: {sorted(used - pretrained)}")

## Step 2: Smoke test / 冒烟测试（约 5 分钟）

1 epoch to verify the pipeline end-to-end **before** the one real training run.

Must-see log lines:
- `vocab : 2545` (pretrained pinyin vocab, NOT a 464-char dataset vocab)
- `Loading checkpoint: pretrained_model_1250000.safetensors` (pretrained weights actually load)
- `Auto warmup: N steps` and a falling loss

In [ ]:
!python -X utf8 src/f5_tts/train/finetune_cli.py --epochs 1 --batch_size_per_gpu 2 --save_per_updates 30 --last_per_updates 60

In [ ]:
# Smoke-test inference: synthesize one sentence with the smoke checkpoint
!python -X utf8 src/f5_tts/infer/infer_cli.py \
    --model_cfg src/f5_tts/configs/F5TTS_v1_Base.yaml \
    --ckpt_file ckpts/child-tts_pinyin/model_last.pt \
    --vocab_file ckpts/child-tts/vocab.txt \
    --ref_audio data/child-tts/000018.wav \
    --ref_text "城里有好多游乐场，可好玩儿了！" \
    --gen_text "今天天气很好，我们去公园玩吧！" \
    --output_dir test_outputs --output_file smoke_test.wav \
    --nfe_step 32 --cfg_strength 2.0

# Intelligibility check: ASR the generated audio and compare with the target text.
# If ASR ≈ target (even if the audio still sounds rough), the pipeline is correct.
import whisper
m = whisper.load_model("small")
print("ASR   :", m.transcribe("test_outputs/smoke_test.wav", language="zh")["text"])
print("Target: 今天天气很好，我们去公园玩吧！")

## Step 3: Full training / 正式训练（一次机会，跑这个）

- 50 epochs, batch 4, lr 1e-5, cosine schedule, auto warmup (10% of total steps)
- `keep_last_n_checkpoints=-1` keeps **every** intermediate checkpoint so the best one can be picked afterwards
- Monitor: `tensorboard --logdir runs`

If OOM: drop `--batch_size_per_gpu` back to 2.

In [ ]:
!python -X utf8 src/f5_tts/train/finetune_cli.py \
    --epochs 50 --batch_size_per_gpu 4 \
    --learning_rate 1e-5 --scheduler cosine \
    --save_per_updates 500 --last_per_updates 100 \
    --keep_last_n_checkpoints -1 --logger tensorboard

## Step 4: Checkpoint selection / 多 checkpoint 选优

For each candidate (`model_500.pt`, `model_1000.pt`, ..., `model_last.pt`):
1. Point `CKPT_FILE` in `batch_inference.py` at the candidate and re-run batch inference
2. Score with `evaluate_quality.py` (PESQ/STOI/DNSMOS) and `evaluate_similarity.py` (SECS)

**Pick by SECS first** (does it sound like the child?), DNSMOS second (naturalness).
Do not pick by training loss — on a 14-minute dataset the lowest-loss checkpoint is usually overfitted.

In [ ]:
!python batch_inference.py        # synthesize the held-out test set
!python evaluate_quality.py       # PESQ / STOI / DNSMOS / ...
!python evaluate_similarity.py    # SECS speaker similarity

## Step 5: Inference / 推理

- CLI: `python speech_synthesis.py "要合成的文本"`
- Web UI: `python app.py` (Gradio, http://localhost:7860)

In [ ]:
from IPython.display import Audio
Audio("test_outputs/smoke_test.wav")